# Chapter 12 -- Durable Execution & Long-Running Agents (Your Working Copy)

Work through this notebook **after reading** `notes/ch12-durable-execution.md`. This chapter takes Chapter 6's plain loop and makes it genuinely durable: a JSONL event log, deterministic replay, and idempotency keys on a side-effecting tool -- then proves it with a **real** kill-and-resume: a standalone script (`durable_agent/agent.py`) is launched as an actual subprocess, deliberately hard-crashed (`os._exit(1)`, indistinguishable from a real `SIGKILL` from the process's own point of view) partway through, and resumed in a fresh process afterward. No simulation of a crash -- the process really dies and a new one picks up the work.

Two exercises below have a stub to fill in: **replaying state from the event log** (notes Section 2) and **making `send_email` exactly-once** (notes Section 4). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- `durable_agent/agent.py`, a Real Standalone Script (Given)

This has to be a **separate process**, not code running inside this notebook's own kernel -- notes Section 2's split (workflow vs. activities) demonstrated for real means the crash has to be real too, and `os._exit(1)` inside this notebook's own kernel would kill the kernel itself, not just simulate a crash. Written to disk once, then driven entirely via `subprocess.run(...)` for the rest of this notebook.

In [ ]:
AGENT_DIR = Path("durable_agent")
AGENT_DIR.mkdir(exist_ok=True)

AGENT_SCRIPT = '''import argparse
import json
import os
import sys


def append_event(log_path, event):
    # Notes Section 2: the event-history log IS the source of truth, not live process memory.
    with open(log_path, "a") as f:
        f.write(json.dumps(event) + "\\n")


def read_completed_steps(log_path):
    # Deterministic replay input: which steps does the log say already happened?
    if not os.path.exists(log_path):
        return set()
    completed = set()
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            event = json.loads(line)
            if event.get("status") == "completed":
                completed.add(event["step"])
    return completed


def send_email(to, idempotency_key, sent_log_path):
    # Notes Section 4: check the idempotency key BEFORE the real effect, not after.
    sent_keys = set()
    if os.path.exists(sent_log_path):
        with open(sent_log_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    sent_keys.add(json.loads(line)["key"])

    if idempotency_key in sent_keys:
        return f"ALREADY_SENT (idempotent no-op) key={idempotency_key}"

    # The real, side-effecting action -- this only ever happens once per key.
    result = f"Email actually sent to {to}"
    with open(sent_log_path, "a") as f:
        f.write(json.dumps({"key": idempotency_key, "to": to}) + "\\n")
    return result


def run_durable(n_steps, event_log_path, sent_log_path, email_step, crash_after_step, crash_after_send_before_log):
    completed = read_completed_steps(event_log_path)
    executed = []
    for step in range(1, n_steps + 1):
        if step in completed:
            continue  # trust the log -- do NOT redo real work for an already-completed step

        executed.append(step)
        if step == email_step:
            result = send_email("user@example.com", idempotency_key=f"welcome-email-step-{step}", sent_log_path=sent_log_path)
            print(f"step {step}: {result}")
            if step == crash_after_send_before_log:
                sys.stdout.flush()
                os._exit(1)  # hard crash AFTER the real send, BEFORE this step is recorded complete
        else:
            print(f"step {step}: did work")

        append_event(event_log_path, {"step": step, "status": "completed"})
        if step == crash_after_step:
            sys.stdout.flush()
            os._exit(1)  # hard crash right after this step's own checkpoint was durably written

    print(f"RUN_COMPLETE executed_steps={executed}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-steps", type=int, default=60)
    parser.add_argument("--event-log", required=True)
    parser.add_argument("--sent-log", required=True)
    parser.add_argument("--email-step", type=int, default=45)
    parser.add_argument("--crash-after-step", type=int, default=None)
    parser.add_argument("--crash-after-send-before-log", type=int, default=None)
    args = parser.parse_args()
    run_durable(args.n_steps, args.event_log, args.sent_log, args.email_step,
                args.crash_after_step, args.crash_after_send_before_log)
'''

(AGENT_DIR / "agent.py").write_text(AGENT_SCRIPT)
print(f"Wrote {AGENT_DIR / 'agent.py'} ({len(AGENT_SCRIPT.splitlines())} lines)")


## Part 2 -- A Real Kill at Step 30, and a Real Resume

`subprocess.run` launches the script as an actual OS process. Passing `--crash-after-step 30` makes it call `os._exit(1)` right after step 30's checkpoint is written -- indistinguishable, from the process's own point of view, from being killed by the OS. The second call **omits** that flag entirely and points at the *same* event log -- exactly what a real resume looks like: no special "resume mode," just re-running the same script against a log that already has some history in it.

In [ ]:
import subprocess
import sys
import json as json_module

RUN_DIR = Path("durable_run")
RUN_DIR.mkdir(exist_ok=True)
EVENT_LOG = RUN_DIR / "events.jsonl"
SENT_LOG = RUN_DIR / "sent_emails.jsonl"

for f in (EVENT_LOG, SENT_LOG):
    if f.exists():
        f.unlink()

print("-" * 60)
print("RUN 1: fresh start, hard-crashing right after step 30")
print("-" * 60)
result_1 = subprocess.run(
    [sys.executable, str(AGENT_DIR / "agent.py"),
     "--n-steps", "60", "--event-log", str(EVENT_LOG), "--sent-log", str(SENT_LOG),
     "--email-step", "45", "--crash-after-step", "30"],
    capture_output=True, text=True,
)
print(result_1.stdout)
print(f"Process exit code: {result_1.returncode}  (nonzero -- it really did os._exit(1), a real process death)")

completed_after_run_1 = set()
with open(EVENT_LOG) as f:
    for line in f:
        completed_after_run_1.add(json_module.loads(line)["step"])
print(f"Event log now shows {len(completed_after_run_1)} completed steps: {sorted(completed_after_run_1)[:5]}...{sorted(completed_after_run_1)[-3:]}")

assert result_1.returncode != 0, "the process should have hard-crashed (nonzero exit)"
assert completed_after_run_1 == set(range(1, 31)), f"expected exactly steps 1-30 completed, got {sorted(completed_after_run_1)}"
print("\nConfirmed: the process really died mid-run, and exactly 30 steps made it into the durable log.")


In [ ]:
print("-" * 60)
print("RUN 2: RESUME -- same event log, no crash flag, no special 'resume' argument at all")
print("-" * 60)
result_2 = subprocess.run(
    [sys.executable, str(AGENT_DIR / "agent.py"),
     "--n-steps", "60", "--event-log", str(EVENT_LOG), "--sent-log", str(SENT_LOG),
     "--email-step", "45"],
    capture_output=True, text=True,
)
print(result_2.stdout)
print(f"Process exit code: {result_2.returncode}")

lines = [l for l in result_2.stdout.splitlines() if l.startswith("step ")]
print(f"\nSteps actually EXECUTED in this second process: {len(lines)} (should be 30, NOT 60)")

assert result_2.returncode == 0, "the resumed run should complete cleanly"
assert len(lines) == 30, f"expected exactly 30 NEW steps executed on resume (31-60), got {len(lines)}"
assert "RUN_COMPLETE executed_steps=" in result_2.stdout
final_completed = set()
with open(EVENT_LOG) as f:
    for line in f:
        final_completed.add(json_module.loads(line)["step"])
assert final_completed == set(range(1, 61)), "the event log should now show all 60 steps completed"

print("\nConfirmed: the SECOND process did not redo steps 1-30 -- it read the event log, saw")
print("they were already there, and only executed the 30 steps that were genuinely missing.")
print("This is a real crash and a real resume, not a simulation of one.")


## Part 3 -- The Idempotency Scenario, For Real: Crash Between the Send and the Record

Notes Section 4's exact hard case: the email genuinely sends, and the process dies **before** that fact gets durably recorded. `--crash-after-send-before-log 10` crashes right there, at step 10, on purpose.

In [ ]:
for f in (EVENT_LOG, SENT_LOG):
    if f.exists():
        f.unlink()

print("-" * 60)
print("RUN 1: crash immediately after send_email fires, BEFORE step 10 is logged complete")
print("-" * 60)
result_3 = subprocess.run(
    [sys.executable, str(AGENT_DIR / "agent.py"),
     "--n-steps", "60", "--event-log", str(EVENT_LOG), "--sent-log", str(SENT_LOG),
     "--email-step", "10", "--crash-after-send-before-log", "10"],
    capture_output=True, text=True,
)
print(result_3.stdout)

sent_after_run_1 = [json_module.loads(l) for l in SENT_LOG.read_text().splitlines() if l.strip()]
completed_after_run_1b = set()
if EVENT_LOG.exists():
    for line in EVENT_LOG.read_text().splitlines():
        if line.strip():
            completed_after_run_1b.add(json_module.loads(line)["step"])

print(f"\nEmails actually sent (real side effects) so far: {len(sent_after_run_1)} -> {sent_after_run_1}")
print(f"Step 10 marked completed in the event log? {10 in completed_after_run_1b}")

assert len(sent_after_run_1) == 1, "the email genuinely sent exactly once, for real, before the crash"
assert 10 not in completed_after_run_1b, "step 10's completion was NOT recorded -- the crash landed exactly in that gap"
print("\nThis is the dangerous gap notes Section 4 describes: the real effect happened, but the")
print("durable record of it happening did not. Resuming naively would re-run step 10 entirely.")


In [ ]:
print("-" * 60)
print("RUN 2: RESUME -- step 10 will run again (the log doesn't know it 'completed')")
print("-" * 60)
result_4 = subprocess.run(
    [sys.executable, str(AGENT_DIR / "agent.py"),
     "--n-steps", "60", "--event-log", str(EVENT_LOG), "--sent-log", str(SENT_LOG),
     "--email-step", "10"],
    capture_output=True, text=True,
)
print(result_4.stdout[:400])
print("...")

sent_after_run_2 = [json_module.loads(l) for l in SENT_LOG.read_text().splitlines() if l.strip()]
print(f"\nEmails actually sent (real side effects) after resume: {len(sent_after_run_2)} -> {sent_after_run_2}")

assert result_4.returncode == 0
assert "ALREADY_SENT" in result_4.stdout, "the idempotency key should have caught the duplicate attempt"
assert len(sent_after_run_2) == 1, f"expected still exactly ONE real email sent despite step 10 re-running, got {len(sent_after_run_2)}"
print("\nConfirmed: step 10 re-ran (it had to -- the log didn't know it finished), send_email")
print("was called a SECOND time with the same idempotency key, and correctly recognized the")
print("real send already happened -- exactly one email went out, not two.")


## Exercise 1 -- Replaying State From the Event Log

Implement `rebuild_state_from_log(log_path)`: notes Section 2's actual point about replay -- reconstruct final state **purely by reading the durable log**, with no re-execution of any step's real logic at all. Given a log of `{"step": N, "status": "completed"}` lines (in the shape `agent.py` writes), return a dict `{"completed_steps": <sorted list of step numbers>, "highest_completed": <int or None>, "next_step": <int>}` -- `next_step` is one past `highest_completed` (or `1` if the log is empty).

In [ ]:
def rebuild_state_from_log(log_path):
    """
    Pure replay: reconstruct progress from the log alone, never by re-running
    any step's actual work. This is notes Section 2's replay principle,
    isolated as its own testable function.
    """
    # TODO: read `log_path` (a Path) if it exists, parse each non-blank line
    # as JSON, and collect every event["step"] where event["status"] ==
    # "completed" into a set. Then build and return:
    #   {"completed_steps": <sorted list>, "highest_completed": <max or None>,
    #    "next_step": <highest_completed + 1, or 1 if there were none>}
    return {"completed_steps": [], "highest_completed": None, "next_step": 1}


In [ ]:
import json
import tempfile

# A hand-built sample log -- deliberately out of order and with a gap, the
# way a real log could look if steps completed out of strict sequence.
sample_log_path = RUN_DIR / "sample_events.jsonl"
sample_log_path.write_text(
    json.dumps({"step": 3, "status": "completed"}) + "\n" +
    json.dumps({"step": 1, "status": "completed"}) + "\n" +
    json.dumps({"step": 2, "status": "completed"}) + "\n"
)

state = rebuild_state_from_log(sample_log_path)
print(f"Rebuilt state: {state}")
assert state["completed_steps"] == [1, 2, 3]
assert state["highest_completed"] == 3
assert state["next_step"] == 4

empty_log_path = RUN_DIR / "empty_events.jsonl"
empty_state = rebuild_state_from_log(empty_log_path)
print(f"Rebuilt state from a nonexistent log: {empty_state}")
assert empty_state == {"completed_steps": [], "highest_completed": None, "next_step": 1}

print("\nExercise 1 PASSED -- state was reconstructed entirely by reading the log, out-of-order")
print("entries sorted correctly, and a missing log correctly implies 'start at step 1'.")


## Exercise 2 -- Making `send_email` Exactly-Once

Implement `idempotent_send_email(to, idempotency_key, sent_log_path)`: the same logic `agent.py` already uses internally, now as a standalone, directly-testable function. Check `sent_log_path` for `idempotency_key` **before** doing anything else; if found, return `f"ALREADY_SENT (idempotent no-op) key={idempotency_key}"` without sending. Otherwise, perform the "real" send (just constructing a result string here), append `{"key": idempotency_key, "to": to}` to `sent_log_path`, and return `f"Email actually sent to {to}"`.

In [ ]:
def idempotent_send_email(to, idempotency_key, sent_log_path):
    """notes Section 4 -- check the key BEFORE the effect, so a retry can never duplicate it."""
    # TODO: read `sent_log_path` (a Path) if it exists, collect every
    # already-used "key" into a set. If `idempotency_key` is already in that
    # set, return f"ALREADY_SENT (idempotent no-op) key={idempotency_key}"
    # WITHOUT sending anything. Otherwise, perform the "real" send (build the
    # result string), append {"key": idempotency_key, "to": to} as a JSON
    # line to sent_log_path, and return the result string.
    return f"Email actually sent to {to}"


In [ ]:
exercise_sent_log = RUN_DIR / "exercise_sent.jsonl"
if exercise_sent_log.exists():
    exercise_sent_log.unlink()

first_call = idempotent_send_email("customer@example.com", "order-4471-confirmation", exercise_sent_log)
second_call = idempotent_send_email("customer@example.com", "order-4471-confirmation", exercise_sent_log)
different_key_call = idempotent_send_email("customer@example.com", "order-9999-confirmation", exercise_sent_log)

print(f"First call:          {first_call}")
print(f"Second call (retry): {second_call}")
print(f"Different key call:  {different_key_call}")

real_sends = [json.loads(l) for l in exercise_sent_log.read_text().splitlines() if l.strip()]
print(f"\nReal sends recorded: {len(real_sends)} -> {real_sends}")

assert first_call == "Email actually sent to customer@example.com"
assert second_call.startswith("ALREADY_SENT")
assert different_key_call == "Email actually sent to customer@example.com"
assert len(real_sends) == 2, f"expected exactly 2 real sends (two DIFFERENT keys), got {len(real_sends)}"
print("\nExercise 2 PASSED -- the same key never sends twice, regardless of how many times it's")
print("retried, while a genuinely different key still sends its own real email.")


## Optional -- Check for a Real Temporal Server

Temporal needs a running server (and typically Docker) to connect to -- not something to spin up inside a notebook by default. This cell checks gracefully rather than assuming, and explains what changes if one's available.

In [ ]:
def check_real_temporal():
    try:
        import temporalio  # noqa: F401
        print("temporalio IS installed -- you could reimplement Part 1's agent as a real")
        print("Temporal Workflow + Activity pair and connect to a local Temporal server.")
    except ImportError:
        print("temporalio is NOT installed (and no Temporal server is assumed running).")
        print("Part 1's durable_agent/agent.py implements notes Section 2's exact split by hand --")
        print("event-history log, replay-by-skipping-completed-steps, idempotency keys -- which is")
        print("structurally what Temporal (or Inngest) provides as a managed service on top of.")


check_real_temporal()


## Key Takeaways

You built a real durable agent as a standalone script, launched it as an actual subprocess, and hard-crashed it on purpose partway through -- not a simulated failure, a real process death via `os._exit(1)`. Resuming meant running the exact same script again with no special flag, and it correctly skipped the 30 already-completed steps, executing only the 30 that were genuinely missing. The idempotency scenario went further: a real side effect (recorded as an actual line in a log) fired, the process died before that fact could be recorded as a completed step, and the resumed run's retry of `send_email` was correctly recognized as a duplicate -- exactly one email went out despite the function being called twice. Both exercises isolated the two mechanisms that made all of this work: pure replay from a log, and an idempotency check that runs before the effect, not after.

**Connection forward:** Chapter 13 leaves single-agent durability behind and asks a cost question one level up -- once an agent can run reliably for hours on its own, when does adding a *second*, isolated agent actually pay for the extra tokens it costs, instead of just being a more complicated way to do the same job.